# EyeAI AMD3 — Run 09 Champion Optimization

This notebook executes the next competition stages without overwriting the current Run 09 checkpoint:

1. **TTA evaluation:** original + horizontal flip, no training.
2. **Run 11:** patient-disjoint 3-fold RETFound training at 224.
3. **OOF analysis:** one prediction per unseen HYAMD development image, one OOF threshold, and an external-positive fold ensemble.
4. **Run 12:** progressive fine-tuning from the saved Run 09 checkpoint at 336.

The locked HYAMD test remains disabled unless explicitly enabled in the final fold-ensemble cell.


## 1. Execution switches and fixed paths

Enable one expensive stage at a time. TTA is inexpensive; cross-validation and 336 fine-tuning train new models.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")

DATASET_MOUNT = Path("/kaggle/input/datasets/alihasan15/hymd-armd-dataset")
DATASET_OVERLAY = Path("/kaggle/working/eyeai_prepared_binary_dataset_retfound")
OUTPUT_ROOT = Path("/kaggle/working/eyeai_binary_ensemble")

# Set an explicit path only when automatic checkpoint discovery finds multiple candidates.
RUN09_CHECKPOINT_OVERRIDE = None
RETFOUND_BASE_CHECKPOINT_OVERRIDE = None

RUN_TTA_EVALUATION = True
RUN_3FOLD_CV = False
BUILD_OOF_RESULTS = False
RUN_PROGRESSIVE_336 = False

# Keep this False until the final model-selection decision is complete.
RUN_FOLD_ENSEMBLE_ON_LOCKED_TEST = False

RESTORE_SAVED_STAGE_OUTPUTS = True
SKIP_COMPLETED_FOLDS = True
FOLD_COUNT = 3


## 2. Clone the repository and install the package

This cell pulls the committed project files and installs the local `eyeai` package.


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Repository and package are ready.")


## 3. Resolve or upgrade the prepared dataset

The saved images are reused. When the attached dataset still has the old manifests, only the CSV/JSON manifests are upgraded in `/kaggle/working`; images are not reprocessed or copied.


In [ ]:
if not DATASET_MOUNT.exists():
    raise FileNotFoundError(f"Attach the Kaggle dataset first: {DATASET_MOUNT}")

def find_source_root(dataset_mount: Path) -> Path:
    candidates = [dataset_mount, dataset_mount / "eyeai_prepared_binary_dataset"]
    for candidate in candidates:
        if (candidate / "dataset_summary.json").exists() and (candidate / "manifests").exists():
            return candidate
    valid = [
        path.parent
        for path in dataset_mount.rglob("dataset_summary.json")
        if (path.parent / "manifests" / "train_mixed.csv").exists()
    ]
    if len(valid) != 1:
        raise RuntimeError(f"Expected one prepared dataset root, found: {valid}")
    return valid[0]

SOURCE_DATASET_ROOT = find_source_root(DATASET_MOUNT)
new_manifests = [
    SOURCE_DATASET_ROOT / "manifests" / "armd_curated_train.csv",
    SOURCE_DATASET_ROOT / "manifests" / "armd_curated_val_positive.csv",
]

if all(path.exists() for path in new_manifests):
    DATASET_ROOT = SOURCE_DATASET_ROOT
    print("Using the upgraded mounted dataset.")
else:
    subprocess.run(
        [
            "python", "-u", "scripts/upgrade_prepared_dataset_manifests.py",
            "--dataset-mount", str(DATASET_MOUNT),
            "--output-root", str(DATASET_OVERLAY),
        ],
        check=True,
        cwd=REPO_DIR,
    )
    DATASET_ROOT = DATASET_OVERLAY

required = [
    DATASET_ROOT / "manifests" / "train_mixed.csv",
    DATASET_ROOT / "manifests" / "hyamd_val.csv",
    DATASET_ROOT / "manifests" / "armd_curated_val_positive.csv",
]
required += [
    DATASET_ROOT / "manifests" / "folds" / f"fold_{fold}_train_mixed.csv"
    for fold in range(FOLD_COUNT)
]
required += [
    DATASET_ROOT / "manifests" / "folds" / f"fold_{fold}_val_hyamd.csv"
    for fold in range(FOLD_COUNT)
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing prepared files:\n" + "\n".join(f"- {path}" for path in missing)
    )

os.environ["EYEAI_DATASET_ROOT"] = str(DATASET_ROOT)
summary = json.loads((DATASET_ROOT / "dataset_summary.json").read_text(encoding="utf-8"))
print("Source dataset root:", SOURCE_DATASET_ROOT)
print("Active dataset root:", DATASET_ROOT)
print(json.dumps(summary, indent=2))


## 4. GPU and checkpoint preflight

Run 09 is loaded from Quick Save output or another attached Kaggle Dataset. The official RETFound CFP checkpoint is also required to construct the architecture safely.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Activate a Kaggle GPU accelerator.")
print("GPU:", torch.cuda.get_device_name(0))

RUN09_FILENAME = "retfound_cfp_run09_last10_mixed_best.pth"
BASE_FILENAME = "RETFound_mae_natureCFP.pth"

def resolve_unique_file(filename: str, override=None, preferred_fragment=None) -> Path:
    if override:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    working = list(Path("/kaggle/working").rglob(filename))
    mounted = list(Path("/kaggle/input").rglob(filename))
    candidates = sorted({path.resolve() for path in working + mounted})
    if preferred_fragment:
        preferred = [path for path in candidates if preferred_fragment in str(path)]
        if len(preferred) == 1:
            return preferred[0]
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/working or /kaggle/input.")
    raise RuntimeError(
        f"Multiple candidates found for {filename}. Set the override variable explicitly:\n"
        + "\n".join(f"- {path}" for path in candidates)
    )

RUN09_CHECKPOINT = resolve_unique_file(
    RUN09_FILENAME,
    override=RUN09_CHECKPOINT_OVERRIDE,
    preferred_fragment="run09_retfound_last10/checkpoints",
)
RETFOUND_BASE_CHECKPOINT = resolve_unique_file(
    BASE_FILENAME,
    override=RETFOUND_BASE_CHECKPOINT_OVERRIDE,
    preferred_fragment="retfound-checkpoint",
)

print("Run 09 checkpoint:", RUN09_CHECKPOINT)
print("RETFound base checkpoint:", RETFOUND_BASE_CHECKPOINT)


## 5. Restore interrupted stage outputs when available

When a previous Quick Save output is attached, this cell restores only Run 11, Run 12, TTA, and OOF stage directories into `/kaggle/working`. This allows `auto_resume` to continue from each fold's `last.pth`.


In [ ]:
def restore_stage_directory(directory_name: str, destination: Path):
    if destination.exists() or not RESTORE_SAVED_STAGE_OUTPUTS:
        return
    matches = [path for path in Path("/kaggle/input").rglob(directory_name) if path.is_dir()]
    preferred = [path for path in matches if "eyeai_binary_ensemble" in str(path)]
    candidates = preferred or matches
    if not candidates:
        return
    if len(candidates) > 1:
        raise RuntimeError(
            f"Multiple saved directories found for {directory_name}:\n"
            + "\n".join(f"- {path}" for path in candidates)
        )
    print(f"Restoring {directory_name} from {candidates[0]}...")
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(candidates[0], destination)

restore_stage_directory(
    "run11_retfound_last10_cv",
    OUTPUT_ROOT / "runs" / "run11_retfound_last10_cv",
)
restore_stage_directory(
    "run12_retfound_last10_progressive336",
    OUTPUT_ROOT / "runs" / "run12_retfound_last10_progressive336",
)
restore_stage_directory(
    "run09_tta",
    OUTPUT_ROOT / "evaluations" / "run09_tta",
)
restore_stage_directory(
    "run11_oof",
    OUTPUT_ROOT / "ensembles" / "run11_oof",
)


## 6. Create resolved runtime configs

The committed YAML files remain portable. This cell injects the actual Kaggle checkpoint paths and the saved Run 09 checkpoint path into temporary configs.


In [ ]:
import yaml

CONFIG_RUN09 = REPO_DIR / "configs/train_retfound_binary_run09_last10_mixed.yaml"
CONFIG_RUN11 = REPO_DIR / "configs/train_retfound_binary_run11_last10_cv.yaml"
CONFIG_RUN12 = REPO_DIR / "configs/train_retfound_binary_run12_progressive336.yaml"

RESOLVED_CONFIG_DIR = Path("/kaggle/working/eyeai_resolved_configs")
RESOLVED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def resolve_config(source_path: Path, run09_checkpoint: Path | None = None) -> Path:
    config = yaml.safe_load(source_path.read_text(encoding="utf-8"))
    config["model"]["retfound_checkpoint_path"] = str(RETFOUND_BASE_CHECKPOINT)
    config["data"]["prepared_dataset_dir"] = str(DATASET_ROOT)
    if run09_checkpoint is not None:
        config["training"]["initial_checkpoint"] = str(run09_checkpoint)
    destination = RESOLVED_CONFIG_DIR / source_path.name
    destination.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    return destination

RESOLVED_RUN09_CONFIG = resolve_config(CONFIG_RUN09)
RESOLVED_RUN11_CONFIG = resolve_config(CONFIG_RUN11)
RESOLVED_RUN12_CONFIG = resolve_config(CONFIG_RUN12, run09_checkpoint=RUN09_CHECKPOINT)

print("Run 09 evaluation config:", RESOLVED_RUN09_CONFIG)
print("Run 11 cross-validation config:", RESOLVED_RUN11_CONFIG)
print("Run 12 progressive-336 config:", RESOLVED_RUN12_CONFIG)


## 7. Stage 1 — TTA evaluation of the current Run 09 checkpoint

No training occurs. The script compares original inference with `original + horizontal flip`, first at Run 09's stored threshold and then at a separately tuned TTA development threshold.


In [ ]:
TTA_OUTPUT_DIR = OUTPUT_ROOT / "evaluations" / "run09_tta"

if RUN_TTA_EVALUATION:
    subprocess.run(
        [
            "python", "-u", "scripts/evaluate_binary_tta.py",
            "--config", str(RESOLVED_RUN09_CONFIG),
            "--checkpoint", str(RUN09_CHECKPOINT),
            "--data-root", str(DATASET_ROOT),
            "--manifest", "manifests/hyamd_val.csv",
            "--variants", "original,hflip",
            "--output-dir", str(TTA_OUTPUT_DIR),
        ],
        check=True,
        cwd=REPO_DIR,
    )
else:
    print("TTA evaluation is disabled.")


## 8. Stage 2 — Run 11: three patient-disjoint folds at 224

Each fold trains a fresh Run 09 architecture from the official RETFound checkpoint. HYAMD patients never overlap between training and validation. External ARMD training images remain train-only. Completed folds are skipped; incomplete folds can auto-resume from their `last.pth` checkpoint when outputs are restored.


In [ ]:
RUN11_ROOT = OUTPUT_ROOT / "runs" / "run11_retfound_last10_cv"

if RUN_3FOLD_CV:
    for fold in range(FOLD_COUNT):
        checkpoint_dir = RUN11_ROOT / "checkpoints" / f"fold_{fold}"
        completed = list(checkpoint_dir.glob("*_best.pth"))
        summary_files = list((RUN11_ROOT / "logs" / f"fold_{fold}").glob("*_summary.json"))
        if SKIP_COMPLETED_FOLDS and completed and summary_files:
            print(f"Fold {fold} is complete; skipping.")
            continue
        print(f"Starting Run 11 fold {fold}...")
        subprocess.run(
            [
                "python", "-u", "scripts/train_binary.py",
                "--config", str(RESOLVED_RUN11_CONFIG),
                "--data-root", str(DATASET_ROOT),
                "--fold", str(fold),
            ],
            check=True,
            cwd=REPO_DIR,
        )
else:
    print("3-fold training is disabled.")


## 9. Stage 3 — OOF predictions, OOF threshold, and external fold ensemble

This stage does not train. It concatenates each fold's patient-unseen validation predictions, verifies exact coverage and no duplicate image IDs, tunes one threshold on all OOF predictions, and averages the three fold predictions on the held-out external-positive set.


In [ ]:
OOF_OUTPUT_DIR = OUTPUT_ROOT / "ensembles" / "run11_oof"

if BUILD_OOF_RESULTS:
    subprocess.run(
        [
            "python", "-u", "scripts/build_oof_ensemble.py",
            "--run-root", str(RUN11_ROOT),
            "--data-root", str(DATASET_ROOT),
            "--output-dir", str(OOF_OUTPUT_DIR),
            "--fold-count", str(FOLD_COUNT),
        ],
        check=True,
        cwd=REPO_DIR,
    )
else:
    print("OOF construction is disabled.")


## 10. Optional final fold ensemble on the locked test

Keep this disabled during development. It averages the three fold models on the locked manifest and applies the OOF-derived threshold. Enable it only after the complete model-selection decision.


In [ ]:
if RUN_FOLD_ENSEMBLE_ON_LOCKED_TEST:
    threshold_json = OOF_OUTPUT_DIR / "run11_oof_threshold.json"
    if not threshold_json.exists():
        raise FileNotFoundError("Build OOF results before locked-test fold inference.")
    subprocess.run(
        [
            "python", "-u", "scripts/predict_fold_ensemble.py",
            "--config", str(RESOLVED_RUN11_CONFIG),
            "--run-root", str(RUN11_ROOT),
            "--data-root", str(DATASET_ROOT),
            "--manifest", "manifests/hyamd_test_locked.csv",
            "--threshold-json", str(threshold_json),
            "--output-dir", str(OUTPUT_ROOT / "final" / "run11_locked_test_fold_ensemble"),
            "--fold-count", str(FOLD_COUNT),
            "--tta", "original,hflip",
        ],
        check=True,
        cwd=REPO_DIR,
    )
else:
    print("Locked-test fold ensemble is disabled.")


## 11. Stage 4 — Run 12: progressive fine-tuning at 336

This is a new training experiment. It starts from the saved Run 09 best checkpoint, interpolates its positional embeddings from 224 to 336, keeps the last 10 blocks trainable, and uses a lower learning rate. The original Run 09 files are never overwritten.


In [ ]:
if RUN_PROGRESSIVE_336:
    subprocess.run(
        [
            "python", "-u", "scripts/train_binary.py",
            "--config", str(RESOLVED_RUN12_CONFIG),
            "--data-root", str(DATASET_ROOT),
        ],
        check=True,
        cwd=REPO_DIR,
    )
else:
    print("Progressive 336 training is disabled.")


## 12. Result dashboard

This cell reads any completed TTA, OOF, Run 11, and Run 12 summaries from `/kaggle/working` without rerunning a stage.


In [ ]:
import pandas as pd

rows = []
for path in sorted(OUTPUT_ROOT.rglob("*_summary.json")):
    try:
        item = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    metrics = item.get("validation_metrics_at_threshold", {}) or item.get("oof_metrics_at_threshold", {}) or item.get("tta_at_tuned_threshold", {}) or {}
    rows.append({
        "artifact": item.get("model_name", path.stem),
        "best_epoch": item.get("best_epoch"),
        "threshold": item.get("best_threshold", item.get("tta_best_threshold", item.get("best_threshold"))),
        "average_precision": item.get("best_selection_score", metrics.get("average_precision")),
        "auc": metrics.get("auc"),
        "macro_f1": metrics.get("macro_f1"),
        "precision_amd": metrics.get("precision_amd"),
        "recall_amd": metrics.get("recall_amd"),
        "specificity": metrics.get("specificity"),
        "path": str(path),
    })

if rows:
    display(pd.DataFrame(rows).sort_values(["macro_f1", "average_precision"], ascending=False, na_position="last"))
else:
    print("No completed optimization summaries were found yet.")
